# Wetland Health Index (WHI) – Del 3: Tröskelvärden och Klassificering
Denna sista notebook i serien används för att applicera den normaliserade modellen från föregående steg på en specifik våtmark och utvärdera den gentemot referensen.

I korthet syftar skriptet till att:
1. **Ladda Våtmarkens Raster:** Ladda in index från det område som ska utvärderas.
2. **Normalisera:** Normalisera pixelvärdena på den *globala skala* (Min/Max) som etablerats i del 2.
3. **Mäta mot Referensen:** Genom att identifiera det percentila intervallet i referensvåtmarken TAM fastställs trösklarna:
    - Mellan 25:e–75:e percentilen = **Good**
    - 5:e–25:e och 75:e–95:e percentilen = **Fair**
    - Utanför 5:e-95:e percentilen = **Poor**
4. **Exportera Klassifikationen:** Spara slutgiltiga `.tif`-filer för GIS.

In [ ]:
import glob
import os
import rioxarray as rxr
import xarray as xr
import numpy as np
import itertools
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from rasterio import open as rio_open
from config import (build_analysis_output_dirs, OPEN_WETLAND_AREA)

### Steg 1: Konfiguration & Studieområde
Laddar bibliotek, väljer vilket studieområde (`area`) och vilket år (`year`) som ska klassificeras i körningen. Sökvägar och clip shape ("öppen våtmark") deklareras direkt därefter.

In [ ]:
# ============================================
# Konfigurera sökvägar och studieområde
# ============================================

area = "LF"  # OBS: här väljs studieområdet
year = 2025  # OBS: här väljs året för variation-data
sensor = "sentinel2"  # "landsat" eller "sentinel2"
ref_area = "TAM"

# clip_shp: sätt till sökväg för att klippa till ett delområde, eller None för hela studieområdet
# Använd: OPEN_WETLAND_AREA[f"{area}_wetland"] för att klippa till den öppna våtmarken
clip_shp = OPEN_WETLAND_AREA[f"{area}_wetland"] # Sätt till None för att inte klippa

clip_name = os.path.basename(clip_shp).replace(".shp", "") if clip_shp is not None else "hela studieområdet"

analysis_output_dirs = build_analysis_output_dirs(sensor=sensor)
output_dir_variation = analysis_output_dirs["variation"]
output_dir_AHP = analysis_output_dirs["AHP"]
os.makedirs(output_dir_variation, exist_ok=True)
os.makedirs(output_dir_AHP, exist_ok=True)

print(f"Studieområde: {area}")
print(f"År: {year}")
print(f"Klippning: {clip_name}")
print(f"Variation-data från: {output_dir_variation}")

### Steg 2: Läs in Index-Data 
Hämtar de underliggande rastren (`EVI_mean`, `MNDWI_mean` och `MNDWI_std`) för området ifrån Del 1. Rastert klipps mot shape-filen så att endast öppen våtmark behålls.

In [ ]:
# =============================================
# Steg 3: Ladda variation-raster från TIF-filer
# =============================================
# Ladda EVI_mean, MNDWI_mean och MNDWI_std för den valda arean och året

index_maps = {}

# Index att ladda: (display_name, file_name, statistik)
index_definitions = [
    ("EVI_mean", "EVI", "mean"),     # Vegetation Index - mean
    ("MNDWI_mean", "MNDWI", "mean"),     # Water Index - mean
    ("MNDWI_std",  "MNDWI", "std"),      # Water Index Variation - standard deviation
]

# Ladda polygon och transformera CRS om nödvändigt
clip_geometry = None
if clip_shp is not None:
    # Hämta rasterns CRS från första filen (läs bara metadata)
    first_filepath = os.path.join(
        output_dir_variation,
        f"EVI_{area}_{year}_mean.tif"
    )
    with rio_open(first_filepath) as src:
        raster_crs = src.crs
    
    # Ladda polygon och transformera CRS om nödvändigt
    aoi = gpd.read_file(clip_shp)
    polygon_crs = aoi.crs
    
    print(f"Polygon CRS: {polygon_crs}")
    print(f"Raster CRS:  {raster_crs}")
    
    if polygon_crs != raster_crs:
        print(f"⚠ CRS skiljer sig! Transformerar polygon från {polygon_crs} till {raster_crs}...")
        aoi = aoi.to_crs(raster_crs)
        print(f"✓ Polygon transformerad\n")
    else:
        print(f"✓ CRS matchar\n")
    
    clip_geometry = aoi.geometry.union_all()
    print(f"Polygon för klippning laddad från: {clip_shp}\n")

for index_key, file_name, stat in index_definitions:
    filepath = os.path.join(
        output_dir_variation, 
        f"{file_name}_{area}_{year}_{stat}.tif"
    )
    
    if os.path.exists(filepath):
        print(f"Laddar {index_key} från: {filepath}")
        data = rxr.open_rasterio(filepath).squeeze("band", drop=True)
        
        # Klipp raster efter polygon om clip_shp är satt
        if clip_shp is not None and clip_geometry is not None:
            data = data.rio.clip([clip_geometry])
            print(f"  ✓ Klippt efter {clip_name}")
        
        index_maps[index_key] = data
        print(f"  ✓ {index_key}: shape={data.shape}, min={float(data.min(skipna=True)):.4f}, max={float(data.max(skipna=True)):.4f}")
    else:
        raise FileNotFoundError(f"Fil saknas: {filepath}")

print(f"\n✓ Alla index laddade för {area} år {year}")

### Steg 3: Global Normalisering
Genom att anropa CSV-filen `global_bounds.csv` från **Del 2** normaliseras rasterna här till referenssystemets globala (våtmarksområden från alla studieområden) Min- och Max-nivåer.

In [ ]:
selected_indices = ["EVI_mean", "MNDWI_mean", "MNDWI_std"]

# =============================================
# Steg 4: Normalisera med globala gränser från WHI_part1
# =============================================
bounds_path = os.path.join(output_dir_AHP, f"global_bounds_{year}.csv")
bounds_df   = pd.read_csv(bounds_path, index_col=0)
global_bounds = bounds_df.to_dict(orient="index")

def minmax_normalize_global(da, global_min, global_max):
    if np.isclose(global_max, global_min):
        return xr.full_like(da, np.nan)
    return (da - global_min) / (global_max - global_min)

# Gemensam valid-mask
valid_mask = xr.full_like(index_maps[selected_indices[0]], True, dtype=bool)
for idx in selected_indices:
    valid_mask = valid_mask & np.isfinite(index_maps[idx])

# Normalisera med globala gränser
normalized = {
    idx: minmax_normalize_global(
        index_maps[idx],
        global_bounds[idx]["min"],
        global_bounds[idx]["max"]
    ).where(valid_mask)
    for idx in selected_indices
}

print("Normaliserade index (globala gränser):")
for idx in selected_indices:
    vals = normalized[idx].values
    print(f"  {idx}: min={np.nanmin(vals):.3f}, max={np.nanmax(vals):.3f}")

### Steg 4: Viktsättning & AHP
Rekonstruerar Analytic Hierarchy Process (AHP)-matrisen på samma vis som i Del 2 för att poängsätta indexen.

| AHP Parvis | MNDWI_mean     | EVI_mean  | MNDWI_std |
|---|---|---|---|
| **MNDWI_mean**       | 1        | 4     | 6      |
| **EVI_mean**       | 1/4        | 1     | 4      |
| **MNDWI_std**     | 1/6        | 1/4     | 1      |

In [ ]:
# Steg 1: Parvis jämförelsematris
# Ordning: MNDWI_mean, EVI_mean, MNDWI_std
matrix = np.array([
    [1,     4,      6],  # MNDWI_mean vs: lika, 4x viktigare, 6x viktigare
    [1/4,   1,      4],  # EVI_mean vs: 1/4, lika, 1/4
    [1/6,   1/4,    1],  # MNDWI_std vs: 1/6, 1/4, lika
])

# Steg 2: Normalisera och beräkna vikter
col_sums = matrix.sum(axis=0)
normalized_matrix = matrix / col_sums
weights = normalized_matrix.mean(axis=1)

index_order = ["MNDWI_mean", "EVI_mean", "MNDWI_std"]
weight_dict = dict(zip(index_order, weights.round(3)))
print("Vikter:", weight_dict)

# Steg 3: Beräkna lambda_max och CR
lambda_max = (matrix @ weights / weights).mean()
n = len(weights)
CI = (lambda_max - n) / (n - 1)
RI = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12}
CR = CI / RI[n]
print(f"λ_max = {lambda_max:.4f}")
print(f"CI    = {CI:.4f}")
print(f"CR    = {CR:.4f} → {'OK' if CR < 0.1 else 'Inkonsekvent!'}")

### Steg 5: Beräkning av AHP Scenarion & Etablera WHI
Summeringen av valda index per pixel sker, till en "WHI score" mellan 0 och 1 för tre viktscenarion (hydrologi, lika viktning, vegetation).

In [ ]:
# =============================================
# Steg 5: Definiera AHP-scenarier och beräkna WHI
# =============================================
# Viktningsscenarier för EVI_mean, NDMI_mean och NDMI_std
# EVI_mean = Vegetation, MNDWI_mean = Water/fuktigher, MNDWI_std = Variabilitet
scenarios = {
    "Equal weighting":      {"EVI_mean": 1/3, "MNDWI_mean": 1/3, "MNDWI_std": 1/3},
    "Hydrology weighting":  {"EVI_mean": weight_dict["EVI_mean"], "MNDWI_mean": weight_dict["MNDWI_mean"], "MNDWI_std": weight_dict["MNDWI_std"]}, # AHP-vikter baserade på expertbedömning
    "Vegetation weighting": {"EVI_mean": 0.671, "MNDWI_mean": 0.244, "MNDWI_std": 0.085}, # AHP-vikter inverterade för att fokusera på vegetation (20260428)
}

results = {}
for scenario_name, w in scenarios.items():
    weight_sum = sum(w.values())
    if not np.isclose(weight_sum, 1.0, atol=0.001):  # Tillåt små avrundningsfel
        raise ValueError(f"Vikterna i '{scenario_name}' summerar till {weight_sum:.3f}, inte 1.0")

    # Beräkning av WHI
    whi = xr.zeros_like(normalized[selected_indices[0]])
    for idx in selected_indices:
        whi = whi + w[idx] * normalized[idx] # Formel: WHI = w1*EVI_mean + w2*MNDWI_mean + w3*MNDWI_std (här blir en ökning i en variabel en ökning av WHI)

    whi = whi.where(valid_mask)
    results[scenario_name] = whi
    print(f"{scenario_name}: medel WHI = {float(np.nanmean(whi.values)):.3f}")

### Steg 6: Fastställa Referenströsklar & Pixelfördelning
Här appliceras själva definitionen av Good/Fair/Poor. Koden:
1. Skannar igenom hela *Referensvåtmarken (TAM)* för motsvarande AHP-scenario.
2. Hittar percentilgränserna (5%, 25%, 75%, 95%) inom dess referensfördelning.
3. Räknar träffytorna gentemot målvåtmarken för att plocka ut hur stor andel av aktuella pixlar som ryms innanför de beräknade intervallen.

Därefter skapas klassningskartorna pixelvis och presenteras.

In [ ]:
# ANVÄNDES INNAN 20260419.

# =============================================
# Steg 6: Referensbaserade klassgränser per scenario
# =============================================

# Ladda TAM WHI per scenario
ref_whi_files = {
    "Equal weighting":  f"TAM_{year}_Lika_viktning_WHI_ref_global.tif",
    "Hydrology weighting":      f"TAM_{year}_AHP_bas_WHI_ref_global.tif",
    "Vegetation weighting": f"TAM_{year}_Vegetationsfokus_WHI_ref_global.tif",
}

scenario_thresholds = {}
for scenario_name, filename in ref_whi_files.items():
    filepath = os.path.join(output_dir_AHP, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"TAM WHI saknas: {filepath}")
    
    whi_ref   = rxr.open_rasterio(filepath).squeeze()
    ref_valid = whi_ref.values[np.isfinite(whi_ref.values)]
    
    
    scenario_thresholds[scenario_name] = {
        "low_thr":   float(np.percentile(ref_valid, 5)),   # Nedre poor-gräns
        "mid_low":   float(np.percentile(ref_valid, 25)),  # Nedre fair-gräns
        "mid_high":  float(np.percentile(ref_valid, 75)),  # Övre fair-gräns
        "high_thr":  float(np.percentile(ref_valid, 95)),  # Övre poor-gräns
        "ref_mean":  float(np.nanmean(ref_valid)),
        "ref_std":   float(np.nanstd(ref_valid)),
        "n_pixels":  len(ref_valid),
    }

    thr = scenario_thresholds[scenario_name]
    print(f"{scenario_name}:")
    print(f"  Poor  < {thr['low_thr']:.3f}  eller  > {thr['high_thr']:.3f}")
    print(f"  Fair  {thr['low_thr']:.3f} – {thr['mid_low']:.3f}  eller  {thr['mid_high']:.3f} – {thr['high_thr']:.3f}")
    print(f"  Good  {thr['mid_low']:.3f} – {thr['mid_high']:.3f}")
    print(f"  Referens mean ± std: {thr['ref_mean']:.3f} ± {thr['ref_std']:.3f}")


# =============================================
# Steg 7: Sammanställ sensitivity-resultat
# =============================================
sensitivity_results = []

for scenario_name, whi_da in results.items():
    # Hämta scenario-specifika trösklar
    thr      = scenario_thresholds[scenario_name]
    low_thr  = thr["low_thr"]
    mid_low  = thr["mid_low"]
    mid_high = thr["mid_high"]
    high_thr = thr["high_thr"]

    whi   = whi_da.values
    valid = np.isfinite(whi)
    total = np.sum(valid)
    if total == 0:
        continue

    good = np.sum((whi >= mid_low)  & (whi <= mid_high) & valid)
    fair = np.sum(((whi >= low_thr) & (whi < mid_low))  & valid) + \
           np.sum(((whi > mid_high) & (whi <= high_thr)) & valid)
    poor = np.sum((whi < low_thr)   & valid) + \
           np.sum((whi > high_thr)  & valid)

    row = {
        "Scenario":               scenario_name,
        "Medel WHI":              round(float(np.nanmean(whi)), 3),
        "Std WHI":                round(float(np.nanstd(whi)), 3),
        "Good (%)":               round(good / total * 100, 1),
        "Fair (%)":               round(fair / total * 100, 1),
        "Poor (%)":               round(poor / total * 100, 1),
        "Referensmetod":          "Symmetriskt percentilintervall (TAM)",
        "TAM ref_mean":           round(thr["ref_mean"], 3),
        "TAM ref_std":            round(thr["ref_std"], 3),
        "low_thr (5:e perc.)":    round(low_thr, 3),
        "mid_low (25:e perc.)":   round(mid_low, 3),
        "mid_high (75:e perc.)":  round(mid_high, 3),
        "high_thr (95:e perc.)":  round(high_thr, 3),
        "TAM ref pixlar":         thr["n_pixels"],
    }

    for idx in selected_indices:
        row[f"Vikt {idx}"] = scenarios[scenario_name][idx]
    sensitivity_results.append(row)

df_sensitivity = pd.DataFrame(sensitivity_results)
print(df_sensitivity)

# =============================================
# Steg 8: Visa klassningskartor (pixelvis)
# =============================================

# Klasskod: 0=NoData, 1=Poor, 2=Fair, 3=Good
def classify_whi(whi_da, low_thr, mid_low, mid_high, high_thr):
    whi = whi_da.values
    cls = np.zeros(whi.shape, dtype=np.uint8)
    valid = np.isfinite(whi)
    # Good: inom referensintervallet (25:e–75:e percentilen)
    cls[(whi >= mid_low)  & (whi <= mid_high) & valid] = 3
    # Fair: i utkanten av referensintervallet
    cls[(whi >= low_thr)  & (whi < mid_low)   & valid] = 2
    cls[(whi > mid_high)  & (whi <= high_thr) & valid] = 2
    # Poor: utanför referensintervallet
    cls[(whi < low_thr)   & valid] = 1
    cls[(whi > high_thr)  & valid] = 1
    return cls

cmap = ListedColormap(["#ffffff", "#d73027", "#fee08b", "#1a9850"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)
legend_handles = [
    Patch(facecolor="#f36860", edgecolor="black", label="Poor"),
    Patch(facecolor="#f4d88b", edgecolor="black", label="Fair"),
    Patch(facecolor="#59de93", edgecolor="black", label="Good"),
    Patch(facecolor="#ffffff", edgecolor="black", label="NoData"),
]

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), constrained_layout=True)
if n == 1:
    axes = [axes]

for ax, (scenario_name, whi_da) in zip(axes, results.items()):
    thr = scenario_thresholds[scenario_name]
    cls = classify_whi(whi_da, thr["low_thr"], thr["mid_low"], thr["mid_high"], thr["high_thr"])
    ax.imshow(cls, cmap=cmap, norm=norm)
    ax.set_title(scenario_name)
    ax.set_axis_off()

fig.suptitle(f"WHI classification with {ref_area} reference ({area}, {year}, {sensor})", fontsize=14)
fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=True)
plt.show()

### Steg 7: Datadistribution & Histogram
Histogram-överlagring (relative frequency) för att ge ett direkt visuellt intryck av hur målvåtmarkens värden är fördelade gentemot referensområdet.

In [ ]:
# =============================================
# Steg 8b: Visualisera WHI-distribution per scenario
# =============================================
from enum import auto

import matplotlib.pyplot as plt
import matplotlib as mpl

# Använd samma stil/fonter som klimatdata_test4
plt.style.use("default")
mpl.rcParams.update({
    "font.family":        "sans-serif",
    #"font.sans-serif":    ["Helvetica", "Arial", "sans-serif"],
    "font.size":          14,
    "axes.titlesize":     14,
    "axes.labelsize":     14,
    "xtick.labelsize":    14,
    "ytick.labelsize":    14,
    "legend.fontsize":    14,
    "axes.spines.top":    False, 
    "axes.spines.right":  False,
    "xtick.direction":    "in",
    "ytick.direction":    "in",
    "xtick.major.width":  1,
    "ytick.major.width":  1,
})

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4), constrained_layout=True)
if len(results) == 1:
    axes = [axes]

# Bestäm ordningen manuellt för presentationen
plot_order = ["Hydrology weighting", "Equal weighting", "Vegetation weighting"]

for i, (ax, scenario_name) in enumerate(zip(axes, plot_order)):
    if scenario_name not in results:
        continue
    whi_da = results[scenario_name]
    thr      = scenario_thresholds[scenario_name]
    low_thr  = thr["low_thr"]
    mid_low  = thr["mid_low"]
    mid_high = thr["mid_high"]
    high_thr = thr["high_thr"]
    ref_mean = thr["ref_mean"]

    # Studieområdets WHI-värden
    vals = whi_da.values[np.isfinite(whi_da.values)]

    # Ladda TAM-fördelning för samma scenario
    tam_filepath = os.path.join(output_dir_AHP, ref_whi_files[scenario_name])
    whi_tam      = rxr.open_rasterio(tam_filepath).squeeze()
    tam_vals     = whi_tam.values[np.isfinite(whi_tam.values)]

    # Plotta båda fördelningarna
    ax.hist(tam_vals, bins=30, color="steelblue", alpha=0.5, 
            label=f"REF (n={len(tam_vals)})",
            weights=np.ones_like(tam_vals) / len(tam_vals) * 100)
    ax.hist(vals, bins=30, color="tomato", alpha=0.5, 
            label=f"HB (n={len(vals)})", 
            weights=np.ones_like(vals) / len(vals) * 100)

# Klassgränser
    ax.axvline(low_thr,  color="#bf2a1adc", linestyle="--", linewidth=1.5,
               label=f"P5") # ({low_thr:.3f})") integrerea denna för att visa värdena också
    ax.axvline(mid_low,  color="#ffae19ff", linestyle="--", linewidth=1.5,
               label=f"P25") # ({mid_low:.3f})")
    ax.axvline(mid_high, color="#ffae19ff", linestyle="--", linewidth=1.5,
               label=f"P75") # ({mid_high:.3f})")
    ax.axvline(high_thr, color="#bf2a1adc", linestyle="--", linewidth=1.5,
               label=f"P95") # ({high_thr:.3f})")
    ax.axvline(ref_mean, color="#385776dc", linestyle="-",  linewidth=1.5,
               label=f"REF mean") #({ref_mean:.3f})")

    # Lägg till pad=10 för att skapa mer luft (ca 10 punkter) mellan diagram och rubrik
    ax.set_title(scenario_name, pad=10)
    ax.set_xlabel("WHI", labelpad=7)
    
    # Sätt fasta intervall på båda axlarna
    ax.set_xlim(0, 0.7)
    
    # Skapa mer utrymme för labels så att '0' och '0.0' inte överlappar
    ax.tick_params(axis='both', which='major', pad=8)
    
    # Tvinga y-axeln att ha ticks vid jämna 2:or, upp till högsta beräknade värdet
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(2))
    
    if i == 0:
        ax.set_ylabel("Relative frequency (%)")
    
    ax.grid(True, linestyle="--", alpha=0.3)

# Skapa gemensam legend centrerad nedanför diagrammen
handles, labels = axes[0].get_legend_handles_labels()
# Genom att sätta y-värdet lägre i bbox_to_anchor (t.ex. -0.12 istället för -0.1) flyttas legenden ner för att ge mer utrymme
fig.legend(handles, labels, loc='outside lower center', ncol=len(handles), bbox_to_anchor=(0.5, -0.12), fontsize=14)

# fig.suptitle(f"WHI-distribution: {area} vs ref ({year})", fontsize=14)
plt.savefig(os.path.join(output_dir_AHP, f"global_{area}_{year}_WHI_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()

### Steg 8: Skapande av Analys-GeoTIFF
Sist i flödet skrivs resultatet ut i `.tif`-format. Filerna (`_WHI` samt `_WHI_class`) innehåller data för visualisering i GIS-program.

In [ ]:
# =============================================
# Steg 9: Spara WHI och klassifikation som GeoTIFF
# =============================================

for scenario_name, whi_da in results.items():
    # Skapa ett filnamns-säkert scenario-namn
    scenario_safe = scenario_name.replace(" ", "_").replace("(", "").replace(")", "")

    # Hämta scenario-specifika trösklar
    thr      = scenario_thresholds[scenario_name]
    low_thr  = thr["low_thr"]
    mid_low  = thr["mid_low"]
    mid_high = thr["mid_high"]
    high_thr = thr["high_thr"]

    # ---- Spara WHI-raster (kontinuerligt 0-1) ----
    whi_out = whi_da.rio.write_crs(whi_da.rio.crs)
    whi_out = whi_out.rio.write_nodata(np.nan)
    whi_path = os.path.join(output_dir_AHP,
                            f"global_{area}_{year}_{scenario_safe}_WHI.tif")
    if os.path.exists(whi_path):
        os.remove(whi_path)
    whi_out.rio.to_raster(whi_path)
    print(f"Sparad WHI: {whi_path}")

    # ---- Spara klassifikationsraster (1=Poor, 2=Fair, 3=Good) ----
    cls = classify_whi(whi_da, low_thr, mid_low, mid_high, high_thr)

    cls_da = xr.DataArray(
        cls.astype("uint8"),
        dims=["y", "x"],
        coords={"y": whi_da.y, "x": whi_da.x}
    )
    cls_da = cls_da.rio.write_crs(whi_da.rio.crs)
    cls_da = cls_da.rio.write_nodata(0)  # 0 = NoData
    cls_path = os.path.join(output_dir_AHP,
                            f"global_{area}_{year}_{scenario_safe}_WHI_class.tif")
    if os.path.exists(cls_path):
        os.remove(cls_path)
    cls_da.rio.to_raster(cls_path)
    print(f"Sparad klassifikation: {cls_path}")

print(f"\n✓ Alla raster sparade i {output_dir_AHP}")